## Step 1: Set up the environment
- 1.1A Install packages
- 1.1B Set up OpenAI Whisper API access
- 1.1C install and verfiy FFmpeg installation


In [ ]:
#1.1A
pip install openai pydub


Note: you may need to restart the kernel to use updated packages.


In [ ]:
#1.1B
%pip show openai
%pip show pydub

Name: openai
Version: 3.46.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License-Expression: Apache-2.0
Location: c:\Users\momin\anaconda3\Lib\site-packages
Requires: anyio, httpx2, jiter, pydantic, sniffio, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: pydub
Version: 0.25.1
Summary: Manipulate audio with an simple and easy high level interface
Home-page: http://pydub.com
Author: James Robert
Author-email: jiaaro@gmail.com
License: MIT
Location: c:\Users\momin\anaconda3\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#1.1C
!ffmpeg -version

ffmpeg version 9.0.2-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
built with gcc 16.2.0 (Rev3, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopencore-

### 1.2 Initialize the OpenAI client and Prepare folders

create three folders:

- `audio` stores the original meeting recording.
- `audio_chunks` stores smaller pieces created from long recordings.
- `transcripts` stores text, subtitle, and JSON exports.

In [2]:
#1.2 Initialize OpenAI client and create necessary directories
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

# Find and load the existing environment configuration.
env_path = find_dotenv(usecwd=True)

if env_path:
    load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY was not found. Check your environment configuration."
    )

# Create the API client without printing or exposing the key.
client = OpenAI()

# Create the folders used by later lab steps.
audio_dir = Path("audio")
chunks_dir = Path("audio_chunks")
transcripts_dir = Path("transcripts")

for folder in (audio_dir, chunks_dir, transcripts_dir):
    folder.mkdir(exist_ok=True)

print("OpenAI client initialized.")
print("Folders prepared:")
print(f"- {audio_dir.resolve()}")
print(f"- {chunks_dir.resolve()}")
print(f"- {transcripts_dir.resolve()}")

OpenAI client initialized.
Folders prepared:
- C:\Users\momin\OneDrive\Desktop\AIPT2026\my first repo computer\my-first-repo\Whisper STT\audio
- C:\Users\momin\OneDrive\Desktop\AIPT2026\my first repo computer\my-first-repo\Whisper STT\audio_chunks
- C:\Users\momin\OneDrive\Desktop\AIPT2026\my first repo computer\my-first-repo\Whisper STT\transcripts


### Step 1 Checkpoint:


- FFmpeg is available.
- openai and pydub are installed.
- The API key was found without exposing it.
- The OpenAI client initialized.
- The required folders were created.

## Step 2: Preparing Sample Meeting Audio
- 2A downloaded  (MP3 so notebook can transcribe)
- 2B saved it in my-first-repo/Whisper STT/audio/
- 2C verify audio file exists and is readable
- 2D if verifiction was not successful,diagnosing the problem
- 2E installing Python 3.14 audio compatibility package (and restart)

In [ ]:
#2A
import os
from urllib.request import urlretrieve 
#downloads a file from a web address and saves it on computer.

os.makedirs("audio", exist_ok=True)

#2B
url = "https://dare.wisc.edu/wp-content/uploads/sites/946/2008/11/ME025.mp3"
urlretrieve(url, "audio/maine_boatmaking.mp3")
#Store this web address under the name url.
#Get the file from url and save it as maine_boatmaking.mp3 inside the audio folder

print("Saved audio/maine_boatmaking.mp3")


Saved audio/maine_boatmaking.mp3


# Additional Step: 
The audio file is present, but Pydub cannot load because its audio-processing dependency is incompatible with the current Python environment. check the Python and Pydub versions before selecting a compatible solution. This diagnostic does not call the API or modify the audio file.

In [ ]:
#2D check versions of Python and Pydub
import sys
from importlib.metadata import version

print("Python version:", sys.version)
print("Pydub version:", version("pydub"))

Python version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Pydub version: 0.25.1


### Install the Python 3.14 audio compatibility package

Python 3.14 no longer includes the older `audioop` module that Pydub 0.25.1 expects.

The `audioop-lts` package provides compatible audio operations so Pydub can load MP3 files while keeping the current Python version unchanged.


In [ ]:
#2.1E install audioop-lts
%pip install audioop-lts

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#2C verfies audio file (does not call API yet- moved down to allow correct installation of audioop-lts)

from pathlib import Path
from pydub import AudioSegment

audio_path = Path("audio/maine_boatmaking.mp3")

if not audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

audio = AudioSegment.from_file(audio_path)

print("Audio file found:", audio_path)
print(f"File size: {audio_path.stat().st_size:,} bytes")
print(f"Duration: {len(audio) / 1000:.2f} seconds")
print(f"Channels: {audio.channels}")
print(f"Sample rate: {audio.frame_rate} Hz")
print(f"Format loaded successfully: {audio_path.suffix}")

Audio file found: audio\maine_boatmaking.mp3
File size: 1,599,572 bytes
Duration: 99.97 seconds
Channels: 2
Sample rate: 44100 Hz
Format loaded successfully: .mp3


### Checkpoint 2: 
- placed audio in  the `audio/` folder so it can be found easily
- verify file exists nd can be read before sending via Whisper API (file path is correct, mp3 is readable, ffmpeg can process the file, audio duration and technical properties are available)


## Step 3: Basic Transcription (Without Chunking)

### 3 Transcribe a short part of the audio file
- Reload API configuration after kernel start (rerun step1.2)
- Send completed sample audio file to Whisper in one API request

In [3]:
from pathlib import Path

audio_path = Path("audio/maine_boatmaking.mp3")

if not audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

print("Sending audio to Whisper...")

with audio_path.open("rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print("Transcription completed.")
print("\nTranscript:")
print("-" * 60)
print(transcript.text)

Sending audio to Whisper...
Transcription completed.

Transcript:
------------------------------------------------------------
Well, how do you go about making a small rowboat? We just make the small scale model and draft it from that. Make a keel out. You make a scale model first? Most everybody does, make a scale model. Or else they draft them out, draw them out on paper. Either one you want to, it doesn't matter. How big are these scale models? The general rule on small type boat is three-quarter inch to a foot. The large ones are up to a quarter inch to a foot. And what's the purpose of the scale model? Well, to determine the length and the width and all this. Oh, I see. They just use smaller everything and then they just scale them up and down. And then how do you go about deciding to build the boat itself? Well, you make a keel first from the model, from the draftings, drawings, whatever it is. Then you make the stem and the stern. And for the small stuff, the small boats where y

### Checkpoint 3
- audio sent to Whisper
- transcription displayed

## Step 4: Prompted transcription — guided approach
- provide Whisper with context prompt describing subject of audio
- save result seperately as the prompted transcription to compare with unprompted result

In [4]:
context_prompt = (
    "This recording discusses Maine boatmaking and may include "
    "nautical, boatbuilding, woodworking, and maritime terminology."
)

print("Sending prompted transcription request to Whisper...")

with audio_path.open("rb") as audio_file:
    prompted_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt
    )

print("Prompted transcription completed.")
print("\nPrompted transcript:")
print("-" * 60)
print(prompted_transcript.text)

Sending prompted transcription request to Whisper...
Prompted transcription completed.

Prompted transcript:
------------------------------------------------------------
How do you go about making a small rowboat? We just make the small scale model and draft it from that. Make a keel out. You make a scale model first? Most everybody does, make a scale model. Or else they draft them out, draw them out on paper. Either one you want to, it doesn't matter. How big are these scale models? The general rule on small type boat is three-quarter inch to a foot. The large ones are up to a quarter inch to a foot. And what's the purpose of the scale model? Well to determine the length and the width and all this. Oh I see, they just use smaller everything and then they just scale them up and down. And then how do you go about deciding to build the boat itself? Well you make a keel first from the model, from the drafting, drawing, whatever it is. Then you make the stem and the stern. And for the smal

## Step 5: Compare Guided and Unguided Transcriptions
compare prompted and unprompted transcripts

In [5]:
unprompted_text = transcript.text
prompted_text = prompted_transcript.text

print("UNPROMPTED TRANSCRIPTION")
print("-" * 60)
print(unprompted_text)

print("\n\nPROMPTED TRANSCRIPTION")
print("-" * 60)
print(prompted_text)

print("\n\nCOMPARISON")
print("-" * 60)
print(f"Unprompted word count: {len(unprompted_text.split())}")
print(f"Prompted word count: {len(prompted_text.split())}")
print(f"Unprompted character count: {len(unprompted_text)}")
print(f"Prompted character count: {len(prompted_text)}")

UNPROMPTED TRANSCRIPTION
------------------------------------------------------------
Well, how do you go about making a small rowboat? We just make the small scale model and draft it from that. Make a keel out. You make a scale model first? Most everybody does, make a scale model. Or else they draft them out, draw them out on paper. Either one you want to, it doesn't matter. How big are these scale models? The general rule on small type boat is three-quarter inch to a foot. The large ones are up to a quarter inch to a foot. And what's the purpose of the scale model? Well, to determine the length and the width and all this. Oh, I see. They just use smaller everything and then they just scale them up and down. And then how do you go about deciding to build the boat itself? Well, you make a keel first from the model, from the draftings, drawings, whatever it is. Then you make the stem and the stern. And for the small stuff, the small boats where you bend the frame, you make a moulse, wha

### Checkpoint 5:
- compared prompted and umprompted transcripts. At this point there was no differnece in the transcription quality accuracy or length.

## Step 6: Split Audio into Chunks

Split the audio into 30-second MP3 files. Each chunk will be transcribed separately and later given a timestamp offset.

In [6]:
from pathlib import Path
from pydub import AudioSegment

audio_path = Path("audio/maine_boatmaking.mp3")
chunks_dir = Path("audio_chunks")
chunks_dir.mkdir(exist_ok=True)

audio = AudioSegment.from_file(audio_path)

chunk_length_ms = 30 * 1000
chunk_records = []

for index, start_ms in enumerate(range(0, len(audio), chunk_length_ms), start=1):
    end_ms = min(start_ms + chunk_length_ms, len(audio))
    chunk = audio[start_ms:end_ms]

    chunk_path = chunks_dir / f"chunk_{index:02d}.mp3"
    chunk.export(chunk_path, format="mp3")

    chunk_records.append({
        "path": chunk_path,
        "start_ms": start_ms,
        "end_ms": end_ms
    })

print(f"Created {len(chunk_records)} chunks.")

for record in chunk_records:
    print(
        f"{record['path']} | "
        f"{record['start_ms'] / 1000:.1f}s–{record['end_ms'] / 1000:.1f}s"
    )

Created 4 chunks.
audio_chunks\chunk_01.mp3 | 0.0s–30.0s
audio_chunks\chunk_02.mp3 | 30.0s–60.0s
audio_chunks\chunk_03.mp3 | 60.0s–90.0s
audio_chunks\chunk_04.mp3 | 90.0s–100.0s


### Checkpoint 6
- successfully created 4 chunks with correct time ranges

## Step 7: Transcribe Chunks with Timestamps

- Transcribe each chunk and add its start-time offset to every returned segment to create timestamps for the original audio timeline.
- Save the response to chun_transcript

In [ ]:
timestamped_segments = [] #Create an empty list called timestamped_segments

for record in chunk_records:
    print(f"Transcribing {record['path']}...")

    with Path(record["path"]).open("rb") as audio_file:
        chunk_transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    chunk_offset = record["start_ms"] / 1000 
    #Take the chunk’s start time in milliseconds and convert it to seconds

    for segment in chunk_transcript.segments:
        timestamped_segments.append({
            "start": chunk_offset + segment.start,
            "end": chunk_offset + segment.end,
            "text": segment.text.strip()
        })

print(f"Collected {len(timestamped_segments)} timestamped segments.")

for segment in timestamped_segments:
    print(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment['text']}")

Transcribing audio_chunks\chunk_01.mp3...
Transcribing audio_chunks\chunk_02.mp3...
Transcribing audio_chunks\chunk_03.mp3...
Transcribing audio_chunks\chunk_04.mp3...
Collected 24 timestamped segments.
[0.00s - 4.00s] Well, how do you go about making a small rowboat?
[4.00s - 11.00s] We just make the small scale model and draft it from that.
[11.00s - 13.00s] Make a keel out.
[13.00s - 16.00s] You make a scale model first?
[16.00s - 19.00s] Most everybody does, make a scale model.
[19.00s - 23.00s] Or else they draft them out, draw them out on paper.
[23.00s - 25.00s] Either one you want to, it doesn't matter.
[25.00s - 28.00s] How big are these scale models?
[28.00s - 30.00s] A general rule on small type.
[30.00s - 35.00s] Oh, just a three-quarter inch to a foot.
[35.00s - 43.50s] The large ones are up to a quarter inch to a foot.
[43.50s - 47.50s] And what's the purpose of the scale model?
[47.50s - 50.50s] Well, to determine the length and the width and all this.
[50.50s - 59.00s] 

### Checkpoint 7
- runs 4 API requests (per chunk)

## Step 8: Export Transcriptions

- Save the timestamped segments as TXT, SRT, and JSON files.
TXT — easy for people to read, search, copy, and review. It contains timestamped text in a simple format.
SRT — used for subtitles and captions in video players. It requires numbered entries with start and end timestamps.
JSON — used by programs and applications. It preserves structured segment data, timestamps, and metadata for later processing. 

In [8]:
from pathlib import Path
import json

transcripts_dir = Path("transcripts")
transcripts_dir.mkdir(exist_ok=True)

def srt_time(seconds):
    total_ms = round(seconds * 1000)
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds, milliseconds = divmod(remainder, 1_000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"

def readable_time(seconds):
    minutes, remainder = divmod(seconds, 60)
    return f"{int(minutes):02}:{remainder:05.2f}"

# TXT export
txt_path = transcripts_dir / "maine_boatmaking_transcript.txt"
txt_lines = [
    f"[{readable_time(segment['start'])} - "
    f"{readable_time(segment['end'])}] {segment['text']}"
    for segment in timestamped_segments
]
txt_path.write_text("\n".join(txt_lines), encoding="utf-8")

# SRT export
srt_path = transcripts_dir / "maine_boatmaking_transcript.srt"
srt_blocks = []

for index, segment in enumerate(timestamped_segments, start=1):
    srt_blocks.append(
        f"{index}\n"
        f"{srt_time(segment['start'])} --> {srt_time(segment['end'])}\n"
        f"{segment['text']}\n"
    )

srt_path.write_text("\n".join(srt_blocks), encoding="utf-8")

# JSON export
json_path = transcripts_dir / "maine_boatmaking_transcript.json"
json_data = {
    "source": "audio/maine_boatmaking.mp3",
    "segments": timestamped_segments
}
json_path.write_text(
    json.dumps(json_data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Export complete:")
print("-", txt_path)
print("-", srt_path)
print("-", json_path)

Export complete:
- transcripts\maine_boatmaking_transcript.txt
- transcripts\maine_boatmaking_transcript.srt
- transcripts\maine_boatmaking_transcript.json
